# Step 2 — calculate block-level building statistics

**# of cells in notebook:** 1 code cell

## Purpose

Calculate building-footprint statistics for every block in `blocks_5`. These statistics describe both the number and size of buildings associated with each block and the amount and density of building footprint area within each block. The resulting metrics are later used by the block-merging algorithm to discourage merges between blocks with substantially different building patterns.

## Input

**Prepared blocks**
- GeoPackage: `E:\_johannesburg\_analysis\segments_v2\blocks_5.gpkg`
- Layer: `blocks_5`

**Selected and projected Overture buildings**
- Geodatabase: `E:\_johannesburg\_analysis\buildings\buildings.gdb`
- Layer: `building_centroid_in_blocks_utm35s`

Required block fields include:

- `block_id`
- `block_area_m2`

## Output

- GeoPackage: `E:\_johannesburg\_analysis\segments_v2\blocks_5_with_building_stats.gpkg`
- Layer: `blocks_5_with_building_stats`

The following building-statistics fields are added:

- `bldg_count`
- `bldg_area_min`
- `bldg_area_max`
- `bldg_area_median`
- `bldg_area_stdev`
- `bldg_area_sum`
- `bldg_area_density`
- `bldg_count_density`

## Main logic

### Cell 1 — calculate and join building statistics to blocks

1. Read the prepared `blocks_5` layer and the selected Overture building polygons.
2. Check required fields, coordinate systems, and geometry validity.
3. Repair invalid geometries with `shapely.make_valid` when needed.
4. Ensure the building layer uses the same projected coordinate system as the blocks.
5. Calculate the full footprint area of each building in square meters.

#### Building-count and building-size statistics

6. Create a guaranteed-inside representative point for each building polygon.
7. Spatially join those points to the block polygons.
8. Assign each matched building to the block containing its representative point.
9. For each block, calculate:
   - number of buildings;
   - minimum building footprint area;
   - maximum building footprint area;
   - median building footprint area;
   - standard deviation of building footprint area.

These statistics use the full area of each building assigned to the block.

#### Building-area totals

10. Separately identify every actual building-polygon/block-polygon intersection.
11. Calculate the area of each intersection.
12. Sum the intersected building area by block to create `bldg_area_sum`.

This means that a building crossing a block boundary contributes only the portion of its footprint lying inside each block when `bldg_area_sum` is calculated.

#### Density metrics and output

13. Join the building statistics back to all blocks.
14. Assign zero building count and zero building-area sum to blocks with no buildings.
15. Calculate:
    - `bldg_area_density = bldg_area_sum / block_area_m2`;
    - `bldg_count_density` as buildings per hectare.
16. Write the completed block layer to `blocks_5_with_building_stats.gpkg`.
17. Print QA totals for:
    - buildings read;
    - buildings assigned by representative point;
    - building count in the output;
    - total intersected building area;
    - blocks containing zero buildings.

The output from this notebook is the principal analytical input to the final block-merging notebook.


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

blocks_gpkg = r"E:\_johannesburg\_analysis\segments_v2\blocks_5.gpkg"
blocks_layer = "blocks_5"

buildings_gdb = r"E:\_johannesburg\_analysis\buildings\buildings.gdb"
buildings_layer = "building_centroid_in_blocks_utm35s"

out_gpkg = r"E:\_johannesburg\_analysis\segments_v2\blocks_5_with_building_stats.gpkg"
out_layer = "blocks_5_with_building_stats"

block_id_field = "block_id"
block_area_field = "block_area_m2"

# Temporary / output building area field
building_area_field = "bldg_area_m2"

# Chunk size for polygon intersection area calculation
chunk_size = 250_000

overwrite_output = True


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def log(msg):
    print(msg, flush=True)


def require_fields(gdf, fields, label):
    missing = [f for f in fields if f not in gdf.columns]
    if missing:
        raise ValueError(
            f"{label} is missing required fields:\n"
            + "\n".join(f"  - {f}" for f in missing)
        )


def make_valid_if_needed(gdf, label):
    invalid = ~gdf.geometry.is_valid
    n_invalid = int(invalid.sum())

    if n_invalid > 0:
        log(f"{label}: fixing {n_invalid:,} invalid geometries with shapely.make_valid...")
        gdf.loc[invalid, "geometry"] = shapely.make_valid(gdf.loc[invalid, "geometry"].array)

    return gdf


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():
    # --------------------------------------------------------
    # Optional: inspect available layers
    # --------------------------------------------------------

    log("Available layers in blocks GeoPackage:")
    print(pyogrio.list_layers(blocks_gpkg))

    log("\nAvailable layers in buildings GDB:")
    print(pyogrio.list_layers(buildings_gdb))

    # --------------------------------------------------------
    # Read blocks
    # --------------------------------------------------------

    log("\nReading blocks...")
    blocks = pyogrio.read_dataframe(
        blocks_gpkg,
        layer=blocks_layer
    )

    require_fields(
        blocks,
        [block_id_field, block_area_field],
        "Blocks layer"
    )

    if blocks.crs is None:
        raise ValueError("Blocks layer has no CRS. Expected UTM 35S.")

    log(f"Blocks read: {len(blocks):,}")
    log(f"Blocks CRS:  {blocks.crs}")

    # Ensure clean index
    blocks = blocks.reset_index(drop=True)
    blocks["block_ix"] = np.arange(len(blocks), dtype=np.int64)

    # Optional geometry validity check
    blocks = make_valid_if_needed(blocks, "Blocks")

    # --------------------------------------------------------
    # Read buildings
    # --------------------------------------------------------

    log("\nReading buildings...")
    buildings = pyogrio.read_dataframe(
        buildings_gdb,
        layer=buildings_layer,
        columns=[],  # geometry only; we calculate area ourselves
    )

    if buildings.crs is None:
        raise ValueError("Buildings layer has no CRS. Expected WGS84.")

    log(f"Buildings read: {len(buildings):,}")
    log(f"Buildings CRS:  {buildings.crs}")

    buildings = buildings.reset_index(drop=True)
    buildings["bldg_ix"] = np.arange(len(buildings), dtype=np.int64)

    # Optional geometry validity check
    buildings = make_valid_if_needed(buildings, "Buildings")

    # --------------------------------------------------------
    # Project buildings to blocks CRS
    # --------------------------------------------------------

    log("\nProjecting buildings to blocks CRS...")
    buildings = buildings.to_crs(blocks.crs)

    # --------------------------------------------------------
    # Calculate full building footprint area in square meters
    # --------------------------------------------------------

    log("Calculating full building footprint area...")
    buildings[building_area_field] = buildings.geometry.area

    # --------------------------------------------------------
    # Create building interior points for centroid-style assignment
    # --------------------------------------------------------
    # representative_point() is equivalent to a guaranteed-inside point.
    # This is usually safer than pure centroid for odd-shaped polygons.
    # --------------------------------------------------------

    log("\nCreating building inside-points...")
    building_points = gpd.GeoDataFrame(
        buildings[["bldg_ix", building_area_field]].copy(),
        geometry=buildings.geometry.representative_point(),
        crs=buildings.crs
    )

    # --------------------------------------------------------
    # Spatial join: building inside-points to blocks
    # --------------------------------------------------------

    log("Spatial joining building points to blocks...")

    blocks_for_join = blocks[[block_id_field, "block_ix", "geometry"]].copy()

    point_join = gpd.sjoin(
        building_points,
        blocks_for_join,
        how="inner",
        predicate="within"
    )

    log(f"Building points matched to blocks: {len(point_join):,}")

    # --------------------------------------------------------
    # Centroid/inside-point based stats
    # --------------------------------------------------------

    log("Calculating centroid/inside-point-based building stats...")

    centroid_stats = (
        point_join
        .groupby(block_id_field)[building_area_field]
        .agg(
            bldg_count="count",
            bldg_area_min="min",
            bldg_area_max="max",
            bldg_area_median="median",
            bldg_area_stdev="std"
        )
        .reset_index()
    )

    # For blocks with only one building, standard deviation is NaN.
    # Store as 0.
    centroid_stats["bldg_area_stdev"] = centroid_stats["bldg_area_stdev"].fillna(0)

    log(f"Blocks with centroid-based stats: {len(centroid_stats):,}")

    # --------------------------------------------------------
    # Polygon-intersection based building area sum
    # --------------------------------------------------------

    log("\nFinding building/block polygon intersections...")

    # Keep only what is needed for the polygon intersection
    buildings_for_itx = buildings[["bldg_ix", "geometry"]].copy()

    candidates = gpd.sjoin(
        buildings_for_itx,
        blocks_for_join,
        how="inner",
        predicate="intersects"
    )

    log(f"Candidate building/block intersections: {len(candidates):,}")

    # Calculate actual intersected area in chunks
    log("Calculating actual intersected building area by block...")

    area_parts = []

    n = len(candidates)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        log(f"  Processing candidates {start:,} to {end:,} of {n:,}...")

        chunk = candidates.iloc[start:end].copy()

        bldg_geoms = chunk.geometry.array

        block_ix = chunk["block_ix"].to_numpy()
        block_geoms = blocks.geometry.iloc[block_ix].array

        inter_geoms = shapely.intersection(bldg_geoms, block_geoms)
        inter_areas = shapely.area(inter_geoms)

        tmp = pd.DataFrame({
            block_id_field: chunk[block_id_field].to_numpy(),
            "itx_area_m2": inter_areas
        })

        area_parts.append(tmp)

    if area_parts:
        area_df = pd.concat(area_parts, ignore_index=True)

        area_stats = (
            area_df
            .groupby(block_id_field)["itx_area_m2"]
            .sum()
            .reset_index()
            .rename(columns={"itx_area_m2": "bldg_area_sum"})
        )
    else:
        area_stats = pd.DataFrame(columns=[block_id_field, "bldg_area_sum"])

    log(f"Blocks with intersected building area: {len(area_stats):,}")

    # --------------------------------------------------------
    # Merge stats back to blocks
    # --------------------------------------------------------

    log("\nMerging stats back to blocks...")

    out = blocks.merge(
        centroid_stats,
        on=block_id_field,
        how="left"
    )

    out = out.merge(
        area_stats,
        on=block_id_field,
        how="left"
    )

    # Fill count and area sum for blocks with no buildings
    out["bldg_count"] = out["bldg_count"].fillna(0).astype("int64")
    out["bldg_area_sum"] = out["bldg_area_sum"].fillna(0.0)

    # Leave min/max/median/stdev as null where no buildings exist,
    # except stdev can also be set to 0 for no-building blocks if preferred.
    # Here I leave no-building stdev as null.
    no_bldg = out["bldg_count"] == 0
    out.loc[no_bldg, "bldg_area_stdev"] = np.nan

    # --------------------------------------------------------
    # Density calculations
    # --------------------------------------------------------

    log("Calculating density fields...")

    block_area = out[block_area_field].astype(float)

    out["bldg_area_density"] = np.where(
        block_area > 0,
        out["bldg_area_sum"] / block_area,
        np.nan
    )

    out["bldg_count_density"] = np.where(
        block_area > 0,
        out["bldg_count"] / (block_area / 10000.0),
        np.nan
    )

    # Remove helper field
    if "block_ix" in out.columns:
        out = out.drop(columns=["block_ix"])

    # --------------------------------------------------------
    # Write output GeoPackage
    # --------------------------------------------------------

    if os.path.exists(out_gpkg):
        if overwrite_output:
            log(f"\nDeleting existing output:\n{out_gpkg}")
            os.remove(out_gpkg)
        else:
            raise FileExistsError(f"Output already exists:\n{out_gpkg}")

    log(f"\nWriting output GeoPackage:\n{out_gpkg}")

    pyogrio.write_dataframe(
        out,
        out_gpkg,
        layer=out_layer,
        driver="GPKG"
    )

    log("")
    log("Done.")
    log(f"Output GeoPackage: {out_gpkg}")
    log(f"Output layer:      {out_layer}")
    log(f"Output features:   {len(out):,}")

    # --------------------------------------------------------
    # Quick QA summary
    # --------------------------------------------------------

    log("")
    log("Quick QA:")
    log(f"Total buildings read:                  {len(buildings):,}")
    log(f"Buildings assigned by inside-point:    {len(point_join):,}")
    log(f"Total building count in output:        {int(out['bldg_count'].sum()):,}")
    log(f"Total intersected building area m2:    {out['bldg_area_sum'].sum():,.2f}")
    log(f"Blocks with zero buildings:            {(out['bldg_count'] == 0).sum():,}")


if __name__ == "__main__":
    main()